# Lab Assignment 9 - NLP using Python
**UCS420 Cognitive Computing**

In [ ]:
import re
import string
import nltk
import pandas as pd
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.probability import FreqDist
from nltk.stem import PorterStemmer, LancasterStemmer, WordNetLemmatizer

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

## Q1. Text preprocessing

In [ ]:
text = ("Cricket is the sport I follow more than anything else, and I have watched it since I was 8 years old. "
        "India lifted the World Cup in 2011, and that final at Wankhede still gives me goosebumps. "
        "A good all-rounder isn't just useful, he changes how a captain plans the whole match. "
        "Bowlers like Jasprit Bumrah keep an economy rate under 5 runs per over, which is incredible in the modern T20 era. "
        "Playing cricket with friends on weekends in our colony ground taught me patience, teamwork and how to lose gracefully. "
        "I still believe a close contest between two strong teams is the best entertainment in the world.")
print(text)

**1. Lowercase and remove punctuation**

In [ ]:
clean_text = text.lower()
clean_text = re.sub(f"[{re.escape(string.punctuation)}]", "", clean_text)
print(clean_text)

**2. Tokenize into words and sentences**

In [ ]:
words = word_tokenize(clean_text)
sentences = sent_tokenize(text)

print("Words:", words)
print("\nTotal words:", len(words))
print("\nSentences:")
for i, s in enumerate(sentences, 1):
    print(i, s)

**3. Remove stopwords**

In [ ]:
stop_words = set(stopwords.words("english"))
filtered_words = [w for w in words if w not in stop_words]

print(filtered_words)
print(f"\n{len(words)} words before, {len(filtered_words)} after removing stopwords")

**4. Word frequency distribution**

In [ ]:
freq = FreqDist(filtered_words)
print(freq.most_common(10))

plt.figure(figsize=(10, 5))
freq.plot(15, title="Top 15 words (stopwords removed)")
plt.show()

## Q2. Stemming and Lemmatization

In [ ]:
porter = PorterStemmer()
lancaster = LancasterStemmer()
lemmatizer = WordNetLemmatizer()

results = pd.DataFrame({
    "word": filtered_words,
    "porter": [porter.stem(w) for w in filtered_words],
    "lancaster": [lancaster.stem(w) for w in filtered_words],
    "lemma (noun)": [lemmatizer.lemmatize(w) for w in filtered_words],
    "lemma (verb)": [lemmatizer.lemmatize(w, pos="v") for w in filtered_words],
})
results

In [ ]:
changed = results[(results["word"] != results["porter"]) | (results["word"] != results["lancaster"]) | (results["word"] != results["lemma (verb)"])]
changed.drop_duplicates("word").reset_index(drop=True)

**Comparison**

- Porter is the milder stemmer, while Lancaster cuts words much harder and often leaves fragments that are not real words.
- Stemmers only chop suffixes, so they can produce non-words that are not in the dictionary.
- The lemmatizer always returns a valid dictionary word, but it needs the correct part of speech. With the default noun setting, verbs like "watched" stay unchanged, and they are only reduced when `pos="v"` is used.
- Stemming is faster and fine for search or indexing. Lemmatization is better when the actual word meaning matters.

## Q3. Regular expressions and text splitting

**Regex extraction**

In [ ]:
long_words = re.findall(r"\b[a-zA-Z]{6,}\b", text)
print("Words with more than 5 letters:")
print(long_words)

In [ ]:
numbers = re.findall(r"\b\d+(?:\.\d+)?\b", text)
print("Numbers:", numbers)

In [ ]:
capitalized = re.findall(r"\b[A-Z][a-zA-Z0-9]*\b", text)
print("Capitalized words:", capitalized)

**Text splitting**

In [ ]:
alpha_words = [w for w in re.split(r"[^A-Za-z]+", text) if w]
print(alpha_words)

In [ ]:
vowel_words = [w for w in alpha_words if w[0].lower() in "aeiou"]
print(vowel_words)

## Q4. Custom tokenization and regex-based cleaning

**Custom tokenizer** (keeps contractions and hyphenated words, keeps decimals intact, numbers are separate tokens)

In [ ]:
def custom_tokenize(s):
    pattern = r"\d+\.\d+|\d+|[A-Za-z]+(?:['’-][A-Za-z]+)*"
    return re.findall(pattern, s)

print(custom_tokenize(text))

In [ ]:
demo = "The state-of-the-art model isn't perfect, but it scored 3.14 on 2 tests and they're 100% sure."
print(custom_tokenize(demo))

**Regex substitutions**

In [ ]:
sample = ("Write to john.doe@example.com or visit https://www.example.com/lab9 for notes. "
          "You can also reach me on 123-456-7890 or +91 9876543210. "
          "Backup mail: support_team@college.edu.in, website www.college.edu.in/help")

email_pattern = r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"
url_pattern = r"(?:https?://|www\.)[^\s,]+"
phone_pattern = r"\+91\s?\d{10}|\b\d{3}-\d{3}-\d{4}\b"

masked = re.sub(email_pattern, "<EMAIL>", sample)
masked = re.sub(url_pattern, "<URL>", masked)
masked = re.sub(phone_pattern, "<PHONE>", masked)

print("Before:\n", sample)
print("\nAfter:\n", masked)